In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
df_check = pd.read_csv("/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_working_sample.tsv", sep="\t", header=0)
print(df_check.columns.tolist())

['score', 'reference', 'paraphrase_1']


In [12]:
%%writefile /content/drive/MyDrive/paraphrase-tool/src/data_prep/style_labeling.py

"""
style_labeling.py
Runs the 500k ParaBank v2 working sample through:
  1. Formality classifier (s-nlp/roberta-base-formality-ranker) -> formal/informal
  2. Politeness classifier (your fine-tuned roberta-base) -> polite/neutral/impolite
Applies the margin-over-chance tie-break rule, balances to ~200k rows
(~33% each), and writes a labeled TSV.
"""

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 2048
MAX_LENGTH = 64

FORMALITY_MODEL_NAME = "s-nlp/roberta-base-formality-ranker"
POLITENESS_CHECKPOINT_PATH = "/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final"

INPUT_TSV = "/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_working_sample.tsv"
OUTPUT_TSV = "/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_labeled.tsv"

POLITENESS_THRESHOLD = 0.5   # P(polite) must exceed this to even be a candidate
TARGET_PER_CLASS = 66_667    # ~200k / 3

# Chance baselines for margin-over-chance normalization
FORMALITY_BASELINE = 0.5      # 2-class
POLITENESS_BASELINE = 1 / 3   # 3-class

# Label scheme for the output column
LABEL_POLITE = 0
LABEL_FORMAL = 1
LABEL_INFORMAL = 2

# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------
print("Loading working sample...")
df = pd.read_csv(
    INPUT_TSV,
    sep="\t",
    header=0,
    quoting=3,  # no quote-char interpretation, matches earlier ParaBank handling
)
print(f"Loaded {len(df):,} rows")

sentences = df["reference"].astype(str).tolist()

# ---------------------------------------------------------------------------
# Batched inference helper
# ---------------------------------------------------------------------------
def run_classifier(sentences, model_name_or_path, id2label):
    """
    Runs a sequence classification model over a list of sentences in batches.
    Returns: list of (predicted_label_str, predicted_label_prob) tuples.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_name_or_path, torch_dtype = torch.float16)
    model.to(DEVICE)
    model.eval()

    results = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc=f"Running {model_name_or_path}"):
            batch = sentences[i:i + BATCH_SIZE]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
            ).to(DEVICE)

            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)
            top_probs, top_idx = probs.max(dim=-1)

            for prob, idx in zip(top_probs.cpu().tolist(), top_idx.cpu().tolist()):
                results.append((id2label[idx], prob))

    del model
    torch.cuda.empty_cache()
    return results

# ---------------------------------------------------------------------------
# Pass 1: Formality
# ---------------------------------------------------------------------------
formality_id2label = {0: "informal", 1: "formal"}
formality_results = run_classifier(sentences, FORMALITY_MODEL_NAME, formality_id2label)
df["formality_label"] = [r[0] for r in formality_results]
df["formality_conf"] = [r[1] for r in formality_results]

# ---------------------------------------------------------------------------
# Pass 2: Politeness
# ---------------------------------------------------------------------------
# NOTE: confirm this ordering matches your training label2id mapping
politeness_id2label = {0: "impolite", 1: "neutral", 2: "polite"}
politeness_results = run_classifier(sentences, POLITENESS_CHECKPOINT_PATH, politeness_id2label)
df["politeness_label"] = [r[0] for r in politeness_results]
df["politeness_conf"] = [r[1] for r in politeness_results]

# ---------------------------------------------------------------------------
# Margins (comparable scale)
# ---------------------------------------------------------------------------
df["formality_margin"] = df["formality_conf"] - FORMALITY_BASELINE
df["politeness_margin"] = df["politeness_conf"] - POLITENESS_BASELINE

# ---------------------------------------------------------------------------
# Label assignment
# ---------------------------------------------------------------------------
def assign_label(row):
    # Default: formality always applies
    default_label = LABEL_FORMAL if row["formality_label"] == "formal" else LABEL_INFORMAL

    # Politeness only competes if it's a real "polite" candidate above threshold
    is_polite_candidate = (
        row["politeness_label"] == "polite"
        and row["politeness_conf"] > POLITENESS_THRESHOLD
    )

    if not is_polite_candidate:
        return default_label

    # Tie-break: whichever margin is higher wins
    if row["politeness_margin"] > row["formality_margin"]:
        return LABEL_POLITE
    else:
        return default_label

print("Assigning final labels...")
df["style_label"] = df.apply(assign_label, axis=1)

# ---------------------------------------------------------------------------
# Check raw distribution before balancing
# ---------------------------------------------------------------------------
print("\nRaw label distribution (pre-balancing):")
print(df["style_label"].value_counts().rename({
    LABEL_POLITE: "polite", LABEL_FORMAL: "formal", LABEL_INFORMAL: "informal"
}))

# ---------------------------------------------------------------------------
# Balance to ~200k total, ~33% per class
# ---------------------------------------------------------------------------
balanced_parts = []
for label_val, name in [(LABEL_POLITE, "polite"), (LABEL_FORMAL, "formal"), (LABEL_INFORMAL, "informal")]:
    subset = df[df["style_label"] == label_val]
    n_available = len(subset)
    if n_available < TARGET_PER_CLASS:
        print(f"WARNING: '{name}' only has {n_available:,} rows, short of target {TARGET_PER_CLASS:,}. "
              f"Will need to pull more from the 7.34M reserve pool.")
    take_n = min(n_available, TARGET_PER_CLASS)
    balanced_parts.append(subset.sample(n=take_n, random_state=42))

balanced_df = pd.concat(balanced_parts).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nFinal balanced dataset: {len(balanced_df):,} rows")
print(balanced_df["style_label"].value_counts().rename({
    LABEL_POLITE: "polite", LABEL_FORMAL: "formal", LABEL_INFORMAL: "informal"
}))

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
output_cols = [
    "reference", "paraphrase_1", "style_label",
    "formality_label", "formality_conf",
    "politeness_label", "politeness_conf",
]
balanced_df[output_cols].to_csv(OUTPUT_TSV, sep="\t", index=False)
print(f"\nSaved to {OUTPUT_TSV}")

Overwriting /content/drive/MyDrive/paraphrase-tool/src/data_prep/style_labeling.py


In [13]:
!python /content/drive/MyDrive/paraphrase-tool/src/data_prep/style_labeling.py

Loading working sample...
Loaded 500,000 rows
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:00<00:00, 1281.71it/s]
Running s-nlp/roberta-base-formality-ranker: 100% 245/245 [06:36<00:00,  1.62s/it]
Loading weights: 100% 201/201 [00:00<00:00, 326.59it/s]
Running /content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final: 100% 245/245 [06:38<00:00,  1.63s/it]
Assigning final labels...

Raw label distribution (pre-balancing):
style_label
formal      275630
informal    202246
polite       22124
Name: count, dtype: int64

Final balanced dataset: 155,458 rows
style_label
formal      66667
informal    66667
polite      22124
Name: count, dtype: int64

Saved to /content/drive/MyDrive/paraphrase-tool/data/processed/parabank_labeled.tsv


In [14]:
!nvidia-smi

Thu Jul  2 16:31:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             32W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [22]:
%%writefile /content/drive/MyDrive/paraphrase-tool/src/data_prep/topup_polite_reserve.py

"""
topup_polite_reserve.py
Pulls fresh rows from the 7.34M-row cleaned reserve pool (rows not already
used in the first 500k working sample), runs them through both classifiers,
keeps only the ones that land as [POLITE], and tops up parabank_labeled.tsv
until the polite bucket hits the 66,667 target.
"""

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ---------------------------------------------------------------------------
# Config (same as style_labeling.py)
# ---------------------------------------------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1024
MAX_LENGTH = 64

FORMALITY_MODEL_NAME = "s-nlp/roberta-base-formality-ranker"
POLITENESS_CHECKPOINT_PATH = "/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final"

CLEANED_POOL_TSV = "/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_cleaned_pool (1).tsv"
LABELED_TSV = "/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_labeled.tsv"

POLITENESS_THRESHOLD = 0.5
POLITE_TARGET = 66_667

FORMALITY_BASELINE = 0.5
POLITENESS_BASELINE = 1 / 3

LABEL_POLITE, LABEL_FORMAL, LABEL_INFORMAL = 0, 1, 2

# Corrected mapping (confirmed against the model's actual config.json)
FORMALITY_ID2LABEL = {0: "informal", 1: "formal"}
POLITENESS_ID2LABEL = {0: "impolite", 1: "neutral", 2: "polite"}

# How many fresh rows to pull from the reserve pool this round.
# ~4.4% of the first 500k landed polite, so aim well above the naive
# estimate to avoid having to run this multiple times.
PULL_SIZE = 1_300_000

# ---------------------------------------------------------------------------
# Load existing labeled data (to know current polite count + what's used)
# ---------------------------------------------------------------------------
print("Loading current labeled dataset...")
labeled_df = pd.read_csv(LABELED_TSV, sep="\t")

current_polite_count = (labeled_df["style_label"] == LABEL_POLITE).sum()
shortfall = POLITE_TARGET - current_polite_count
print(f"Current polite count: {current_polite_count:,} | Shortfall: {shortfall:,}")

if shortfall <= 0:
    print("Polite bucket already at target. Nothing to do.")
    raise SystemExit

used_references = set(labeled_df["reference"])

# ---------------------------------------------------------------------------
# Load reserve pool, exclude already-used rows, sample a fresh batch
# ---------------------------------------------------------------------------
print("Loading cleaned reserve pool (this may take a while, it's a big file)...")
pool_df = pd.read_csv(CLEANED_POOL_TSV, sep="\t")
print(f"Reserve pool size: {len(pool_df):,}")

pool_df = pool_df[~pool_df["reference"].isin(used_references)]
print(f"Reserve pool after excluding already-used rows: {len(pool_df):,}")

pull_size = min(PULL_SIZE, len(pool_df))
batch_df = pool_df.sample(n=pull_size, random_state=42).reset_index(drop=True)
print(f"Pulled {len(batch_df):,} fresh rows for this round")

sentences = batch_df["reference"].astype(str).tolist()

# ---------------------------------------------------------------------------
# Batched inference helper (fp16, same as style_labeling.py)
# ---------------------------------------------------------------------------
def run_classifier(sentences, model_name_or_path, id2label):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path, torch_dtype=torch.float16
    )
    model.to(DEVICE)
    model.eval()

    results = []
    with torch.no_grad():
        for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc=f"Running {model_name_or_path}"):
            batch = sentences[i:i + BATCH_SIZE]
            encoded = tokenizer(
                batch, padding=True, truncation=True,
                max_length=MAX_LENGTH, return_tensors="pt",
            ).to(DEVICE)

            logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1)
            top_probs, top_idx = probs.max(dim=-1)

            for prob, idx in zip(top_probs.float().cpu().tolist(), top_idx.cpu().tolist()):
                results.append((id2label[idx], prob))

    del model
    torch.cuda.empty_cache()
    return results

# ---------------------------------------------------------------------------
# Run both classifiers
# ---------------------------------------------------------------------------
formality_results = run_classifier(sentences, FORMALITY_MODEL_NAME, FORMALITY_ID2LABEL)
batch_df["formality_label"] = [r[0] for r in formality_results]
batch_df["formality_conf"] = [r[1] for r in formality_results]

politeness_results = run_classifier(sentences, POLITENESS_CHECKPOINT_PATH, POLITENESS_ID2LABEL)
batch_df["politeness_label"] = [r[0] for r in politeness_results]
batch_df["politeness_conf"] = [r[1] for r in politeness_results]

# ---------------------------------------------------------------------------
# Margins + tie-break (identical logic to style_labeling.py)
# ---------------------------------------------------------------------------
batch_df["formality_margin"] = batch_df["formality_conf"] - FORMALITY_BASELINE
batch_df["politeness_margin"] = batch_df["politeness_conf"] - POLITENESS_BASELINE

def assign_label(row):
    default_label = LABEL_FORMAL if row["formality_label"] == "formal" else LABEL_INFORMAL
    is_polite_candidate = (
        row["politeness_label"] == "polite"
        and row["politeness_conf"] > POLITENESS_THRESHOLD
    )
    if not is_polite_candidate:
        return default_label
    if row["politeness_margin"] > row["formality_margin"]:
        return LABEL_POLITE
    return default_label

batch_df["style_label"] = batch_df.apply(assign_label, axis=1)

# ---------------------------------------------------------------------------
# Keep only new polite rows, trim to exactly what's needed, append, save
# ---------------------------------------------------------------------------
new_polite = batch_df[batch_df["style_label"] == LABEL_POLITE]
print(f"\nNew polite rows found this round: {len(new_polite):,}")

if len(new_polite) < shortfall:
    print(f"WARNING: still short by {shortfall - len(new_polite):,} rows. "
          f"Rerun with a larger PULL_SIZE or a different random_state to pull more.")

take_n = min(len(new_polite), shortfall)
new_polite = new_polite.sample(n=take_n, random_state=42)

output_cols = [
    "reference", "paraphrase_1", "style_label",
    "formality_label", "formality_conf",
    "politeness_label", "politeness_conf",
]
final_df = pd.concat([labeled_df[output_cols], new_polite[output_cols]], ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)

final_df.to_csv(LABELED_TSV, sep="\t", index=False)

print(f"\nFinal dataset: {len(final_df):,} rows")
print(final_df["style_label"].value_counts().rename({
    LABEL_POLITE: "polite", LABEL_FORMAL: "formal", LABEL_INFORMAL: "informal"
}))

Overwriting /content/drive/MyDrive/paraphrase-tool/src/data_prep/topup_polite_reserve.py


In [23]:
!python /content/drive/MyDrive/paraphrase-tool/src/data_prep/topup_polite_reserve.py

Loading current labeled dataset...
Current polite count: 22,124 | Shortfall: 44,543
Loading cleaned reserve pool (this may take a while, it's a big file)...
Reserve pool size: 7,338,992
Reserve pool after excluding already-used rows: 7,189,699
Pulled 1,300,000 fresh rows for this round
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 201/201 [00:00<00:00, 972.48it/s]
Running s-nlp/roberta-base-formality-ranker: 100% 1270/1270 [16:24<00:00,  1.29it/s]
Loading weights: 100% 201/201 [00:00<00:00, 304.62it/s]
Running /content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier-v3/final: 100% 1270/1270 [16:39<00:00,  1.27it/s]

New polite rows found this round: 55,383

Final dataset: 200,001 rows
style_label
informal    66667
polite      66667
formal      66667
Name: count, dtype: int64


In [21]:
import os
path = "/content/drive/MyDrive/paraphrase-tool/checkpoints/politeness-classifier/final"
print(os.path.exists(path))
print(os.listdir(path) if os.path.exists(path) else "NOT FOUND")

False
NOT FOUND


In [17]:
import os
folder = "/content/drive/MyDrive/paraphrase-tool/data/processed/"
print(os.listdir(folder))

['parabank_working_sample.tsv', 'parabank_cleaned_pool (1).tsv', 'parabank_labeled.tsv']


In [18]:
import os
path = "/content/drive/MyDrive/paraphrase-tool/data/processed/parabank_cleaned_pool (1).tsv"
print(f"{os.path.getsize(path) / (1024**3):.2f} GB")

1.12 GB
